# 2. Model Training & Forecasting

Trains RandomForest, XGBoost, and LightGBM on the simulated IoT dataset, compares them, and forecasts the next 7 days of concrete demand. Shared logic lives in `src/train_model.py`.

> Run `01_IOT_Simulation.ipynb` first (or `python -m src.iot_simulation`) to generate `../data/IoT_Simulation_Output.csv`.

In [ ]:
import sys
sys.path.append('..')

import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.train_model import train_and_compare, pick_best_model, forecast_next_days


## 1. Load the simulated dataset

In [ ]:
df = pd.read_csv('../data/IoT_Simulation_Output.csv')
print('Rows:', len(df))
df.head()


## 2. Train and compare models

In [ ]:
trained_models, results = train_and_compare(df)

for name, metrics in results.items():
    print(f"{name:<15} R2: {metrics['r2']:.4f} | MAE: {metrics['mae']:.4f}")


## 3. Pick the best model

In [ ]:
best_name, best_model = pick_best_model(trained_models, results)
print('Best model:', best_name)


## 4. Model comparison chart

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(results.keys(), [m['r2'] for m in results.values()])
plt.title('Model Comparison (R2 Score)')
plt.ylabel('R2 Score')
plt.show()


## 5. Forecast the next 7 days

In [ ]:
zones = df['Zone'].unique().tolist()
forecast = forecast_next_days(best_model, df, zones=zones, days=7, seed=1)
forecast_display = forecast[['Date', 'Zone', 'Predicted_Daily_Usage_m3']]
forecast_display


In [ ]:
total_demand = forecast['Predicted_Daily_Usage_m3'].sum()
print(f'Total concrete needed over next 7 days: {total_demand:.2f} m3')


## 6. Save the trained model

Saved so `app/app.py` can load it directly instead of retraining on every run.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
print('Saved ../models/best_model.pkl (%s)' % best_name)
